In [2]:
import pandas as pd

file_path = '/content/Smart_Farming_Crop_Yield_2024.csv'

try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded data from {file_path}.")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file at '{file_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred while loading the file: {e}")

Successfully loaded data from /content/Smart_Farming_Crop_Yield_2024.csv.


,farm_id,region,crop_type,soil_moisture_%,soil_pH,temperature_C,rainfall_mm,humidity_%,sunlight_hours,irrigation_type,...,sowing_date,harvest_date,total_days,yield_kg_per_hectare,sensor_id,timestamp,latitude,longitude,NDVI_index,crop_disease_status
0,FARM0001,North India,Wheat,35.95,5.99,17.79,75.62,77.03,7.27,NaN,...,2024-01-08,2024-05-09,122,4408.07,SENS0001,2024-03-19,14.970941,82.997689,0.63,Mild
1,FARM0002,South USA,Soybean,19.74,7.24,30.18,89.91,61.13,5.67,Sprinkler,...,2024-02-04,2024-05-26,112,5389.98,SENS0002,2024-04-21,16.613022,70.869009,0.58,NaN
2,FARM0003,South USA,Wheat,29.32,7.16,27.37,265.43,68.87,8.23,Drip,...,2024-02-03,2024-06-26,144,2931.16,SENS0003,2024-02-28,19.503156,79.068206,0.80,Mild
3,FARM0004,Central USA,Maize,17.33,6.03,33.73,212.01,70.46,5.03,Sprinkler,...,2024-02-21,2024-07-04,134,4227.80,SENS0004,2024-05-14,31.071298,85.519998,0.44,NaN
4,FARM0005,Central USA,Cotton,19.37,5.92,33.86,269.09,55.73,7.93,NaN,...,2024-02-05,2024-05-20,105,4979.96,SENS0005,2024-04-13,16.568540,81.691720,0.84,Severe


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   farm_id               500 non-null    object 
 1   region                500 non-null    object 
 2   crop_type             500 non-null    object 
 3   soil_moisture_%       500 non-null    float64
 4   soil_pH               500 non-null    float64
 5   temperature_C         500 non-null    float64
 6   rainfall_mm           500 non-null    float64
 7   humidity_%            500 non-null    float64
 8   sunlight_hours        500 non-null    float64
 9   irrigation_type       350 non-null    object 
 10  fertilizer_type       500 non-null    object 
 11  pesticide_usage_ml    500 non-null    float64
 12  sowing_date           500 non-null    object 
 13  harvest_date          500 non-null    object 
 14  total_days            500 non-null    int64  
 15  yield_kg_per_hectare  5

In [4]:
missing_values = df.isnull().sum()
display(missing_values[missing_values > 0])

,0
irrigation_type,150
crop_disease_status,130


In [5]:
# Impute missing values in 'irrigation_type' with its mode
mode_irrigation_type = df['irrigation_type'].mode()[0]
df['irrigation_type'].fillna(mode_irrigation_type, inplace=True)
print(f"Missing values in 'irrigation_type' imputed with: {mode_irrigation_type}")

# Impute missing values in 'crop_disease_status' with its mode
mode_crop_disease_status = df['crop_disease_status'].mode()[0]
df['crop_disease_status'].fillna(mode_crop_disease_status, inplace=True)
print(f"Missing values in 'crop_disease_status' imputed with: {mode_crop_disease_status}")

# Verify that missing values have been handled
print("\nMissing values after imputation:")
display(df[['irrigation_type', 'crop_disease_status']].isnull().sum())

Missing values in 'irrigation_type' imputed with: Sprinkler
Missing values in 'crop_disease_status' imputed with: Severe

Missing values after imputation:


/tmp/ipykernel_580/2252456067.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['irrigation_type'].fillna(mode_irrigation_type, inplace=True)
/tmp/ipykernel_580/2252456067.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplac

,0
irrigation_type,0
crop_disease_status,0


In [6]:
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Number of duplicate rows found: {duplicate_rows}")
    # Optionally, display the duplicate rows
    # display(df[df.duplicated(keep=False)])
else:
    print("No duplicate rows found in the DataFrame.")

No duplicate rows found in the DataFrame.


### 1. Convert Date Columns and Create `growth_duration` Feature

In [7]:
import pandas as pd

# Convert 'sowing_date' and 'harvest_date' to datetime objects
df['sowing_date'] = pd.to_datetime(df['sowing_date'])
df['harvest_date'] = pd.to_datetime(df['harvest_date'])

# Calculate 'growth_duration' in days
df['growth_duration'] = (df['harvest_date'] - df['sowing_date']).dt.days

# Convert 'timestamp' to datetime objects (if it represents a single event time)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print("Date columns converted and 'growth_duration' feature created.")
display(df[['sowing_date', 'harvest_date', 'timestamp', 'growth_duration']].head())

Date columns converted and 'growth_duration' feature created.


,sowing_date,harvest_date,timestamp,growth_duration
0,2024-01-08,2024-05-09,2024-03-19,122
1,2024-02-04,2024-05-26,2024-04-21,112
2,2024-02-03,2024-06-26,2024-02-28,144
3,2024-02-21,2024-07-04,2024-05-14,134
4,2024-02-05,2024-05-20,2024-04-13,105


### 2. Categorical Feature Encoding

Many machine learning models require numerical input. Categorical features like `region`, `crop_type`, `irrigation_type`, `fertilizer_type`, and `crop_disease_status` need to be converted. One-hot encoding is a common method for this.

I will now apply one-hot encoding to these columns. This will create new binary columns for each category.

In [8]:
# Identify categorical columns to encode (excluding those that might be unique identifiers or already handled)
categorical_cols = ['region', 'crop_type', 'irrigation_type', 'fertilizer_type', 'crop_disease_status']

# Apply one-hot encoding
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Categorical features one-hot encoded.")
display(df.head())

Categorical features one-hot encoded.


,farm_id,soil_moisture_%,soil_pH,temperature_C,rainfall_mm,humidity_%,sunlight_hours,pesticide_usage_ml,sowing_date,harvest_date,...,crop_type_Maize,crop_type_Rice,crop_type_Soybean,crop_type_Wheat,irrigation_type_Manual,irrigation_type_Sprinkler,fertilizer_type_Mixed,fertilizer_type_Organic,crop_disease_status_Moderate,crop_disease_status_Severe
0,FARM0001,35.95,5.99,17.79,75.62,77.03,7.27,6.34,2024-01-08,2024-05-09,...,False,False,False,True,False,True,False,True,False,False
1,FARM0002,19.74,7.24,30.18,89.91,61.13,5.67,9.60,2024-02-04,2024-05-26,...,False,False,True,False,False,True,False,False,False,True
2,FARM0003,29.32,7.16,27.37,265.43,68.87,8.23,15.26,2024-02-03,2024-06-26,...,False,False,False,True,False,False,True,False,False,False
3,FARM0004,17.33,6.03,33.73,212.01,70.46,5.03,25.80,2024-02-21,2024-07-04,...,True,False,False,False,False,True,False,True,False,True
4,FARM0005,19.37,5.92,33.86,269.09,55.73,7.93,25.65,2024-02-05,2024-05-20,...,False,False,False,False,False,True,True,False,False,True


### Save Cleaned Dataset

Now that the data has been cleaned and preprocessed, we can save it to a new CSV file for future use. This will preserve all the transformations we've applied.

In [ ]:
output_file_path = '/content/cleaned_smart_farming_crop_yield.csv'
df.to_csv(output_file_path, index=False)
print(f"Cleaned dataset saved to: {output_file_path}")